In [2]:
import os
import torch

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("has xpu:", hasattr(torch, "xpu"))
if hasattr(torch, "xpu"):
    print("xpu available:", torch.xpu.is_available())

# DirectML is optional (Windows DX12). Avoid noisy output unless you opted in.
_dml_env = os.environ.get("TORCH_DEVICE", "").strip().lower() in ("directml", "dml")
_use_dml = os.environ.get("USE_DIRECTML", "").strip().lower() in ("1", "true", "yes", "on")
_dml_debug = os.environ.get("TORCH_DEVICE_DEBUG", "").strip().lower() in ("1", "true", "yes", "on")
if _use_dml or _dml_env or _dml_debug:
    try:
        import torch_directml as dml  # type: ignore

        print("directml ok:", dml.device())
    except Exception as e:
        print("directml not ready:", e)

torch: 2.9.1+xpu
cuda: False
has xpu: True
xpu available: True


In [3]:
import sys
import os

# 첫 셀과 아래 `torch.__version__`이 다르면(한쪽 +xpu, 한쪽 +cpu) 노트북에 남은 예전 출력입니다.
# Kernel → Restart 후 이 셀에 찍힌 버전만 기준으로 보세요. +cpu면 `uv sync --no-default-groups --group xpu` 필요.

# Skip Huawei Ascend `torch.npu` in auto device order (set TORCH_SKIP_NPU=0 to allow).
os.environ.setdefault("TORCH_SKIP_NPU", "1")

# --- Speed knobs (all opt-in, default 미설정 = 기존 동작 보존) -------------
# 측정값: examples/sslgraph/bench/profile_pretrain.py (ESOL/batch=400/XPU).
#
#   MMFFRANDOM_FAST=1     -> views_fn 의 'MMFFrandom' 을 PyG to_data_list/from_data_list
#                            대신 torch.index_select 한 번으로 (614ms -> 1.8ms / iter).
#                            출력 분포(2개 MMFF 슬롯 uniform) 동일.
#   PRETRAIN_AMP=bf16|fp16 -> forward+loss 영역을 autocast. XPU bf16 ~1.9x.
#   DATALOADER_NUM_WORKERS=N -> background workers + persistent_workers.
#   PIN_MEMORY=1          -> pin_memory=True (H2D 복사 가속).
#
# 켜고 싶은 것만 위 셀들 실행 전에 os.environ[...] = '...' 로 세팅.
# 전후 비교(막대·시계열·JSON, 호스트 env 비파괴): examples/sslgraph/bench/run_pretrain_compare.ipynb

# If a previous `import torch` failed mid-flight, Python can keep a *broken*
# `torch` in sys.modules — the next import returns that stub (NameError: `_C`).
# Only strip those stubs here. Never delete a working torch and re-import in the
# same Jupyter kernel: that can raise RuntimeError: `_has_torch_function` already has a docstring.
_torch_mod = sys.modules.get("torch")
if _torch_mod is not None and not hasattr(_torch_mod, "_C"):
    for _k in list(sys.modules.keys()):
        if _k == "torch" or _k.startswith("torch."):
            del sys.modules[_k]

# Wrong kernel → PyTorch init may fail similarly. Fix kernel or:
# uv run python -m ipykernel install --user --name 3dgcl --display-name "3dgcl (uv)"
if ".venv" not in sys.executable.replace("/", "\\"):
    raise RuntimeError(
        "커널이 프로젝트 .venv 가 아닙니다. 현재 실행 파일:\n"
        + sys.executable
    )

# Windows CPU venv + conda base: preload torch DLL dir before native code loads (DLL search order mixups).
if sys.platform == "win32":
    try:
        from pathlib import Path

        _venv = Path(sys.executable).resolve().parent.parent
        _torch_lib = _venv / "Lib" / "site-packages" / "torch" / "lib"
        if _torch_lib.is_dir():
            os.add_dll_directory(str(_torch_lib))
            _pb = sys.base_exec_prefix
            _conda_bin = os.path.join(_pb, "Library", "bin") if _pb != sys.prefix else ""
            if _conda_bin and os.path.isdir(_conda_bin):
                os.add_dll_directory(_conda_bin)
    except OSError:
        pass

import torch

print(torch.__version__)
print("cuda available:", torch.cuda.is_available())
if "+cpu" in torch.__version__.lower():
    print(
        "[hint] 이 환경은 CPU 전용 torch(+cpu)입니다. Intel XPU를 쓰려면 "
        "`uv sync --no-default-groups --group xpu` 후 커널을 재시작하세요."
    )
xm = getattr(torch, "xpu", None)
if xm is not None:
    xi = (os.environ.get("XPU_DEVICE_INDEX", "0") or "0").strip()
    ia, dc = xm.is_available(), xm.device_count()
    print(f"xpu is_available={ia} device_count={dc}")
    # pick_torch_device()는 드물게 여기까지 False여도 빠져 CPU가 됩니다. 명시 디바이스로 고정.
    if "+cpu" not in torch.__version__.lower() and not (
        os.environ.get("TORCH_DEVICE") or ""
    ).strip():
        os.environ.setdefault("TORCH_DEVICE", f"xpu:{xi}")
        print(f"[setup] TORCH_DEVICE default -> xpu:{xi} (직접 끄려면 TORCH_DEVICE=cpu)")


2.9.1+xpu
cuda available: False
xpu is_available=True device_count=1
[setup] TORCH_DEVICE default -> xpu:0 (직접 끄려면 TORCH_DEVICE=cpu)


In [4]:
from IPython.display import display, HTML

display(HTML("<style>.container { width:90% !important; }</style>"))

import sys

sys.path.insert(0, "..")
sys.path.insert(0, "../..")

import pandas as pd
import matplotlib.pyplot as plt
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")

# `import torch` only in cell 0 — re-import here can resurrect a stale partial torch module.
from torch.utils.tensorboard import SummaryWriter

from dig.sslgraph.utils import Encoder
from dig.sslgraph.utils.device import pick_torch_device
from dig.sslgraph.evaluation import Pretrain
from dig.threedgraph.dataset import MoleculeNet, QM
from dig.sslgraph.method import GraphCL

print("pick_torch_device() ->", pick_torch_device())



pick_torch_device() -> xpu:0


In [ ]:
# 위쪽 순서대로 실행하는 게 가장 안전합니다. 위 import 셀을 건너뛰면 `NameError`가 납니다.
# 이 블록을 두어 같은 셀에서도 최소 분량의 경로/import가 준비되게 했습니다.
import sys

sys.path.insert(0, "..")
sys.path.insert(0, "../..")

from dig.sslgraph.utils.device import pick_torch_device
from dig.sslgraph.utils import Encoder
from dig.sslgraph.evaluation import Pretrain
from dig.sslgraph.method import GraphCL


def main():
    import argparse as _argparse

    _parser = _argparse.ArgumentParser()
    args = _parser.parse_args([])

    # Finetune or rand init
    args.finetune = False
    args.seed = 2222

    # File Path
    args.model_path = './models'

    # Device — see dig.sslgraph.utils.device.pick_torch_device (CUDA→MPS→XPU→NPU→DirectML→CPU)
    args.device = pick_torch_device()
    print("args.device:", args.device)

    # Dataset
    args.pretrain_dataset = 'esol'
    args.batch_size = 400
    
    # Model
    args.encoder = 'schnet'
    args.edge_weight = True
    args.feat_dim = 9
    args.cutoff = 5.0  # [5.0, 10.0]
    args.num_layers = 2 # [2, 4]
    args.num_filters = 128
    args.num_gaussians = 50
    args.z_dim = 32
    
    args.int_emb_size = 64
    args.basis_emb_size_dist = 8
    args.basis_emb_size_angle = 8
    args.basis_emb_size_torsion = 8
    args.out_emb_channels = 256
    args.num_spherical = 3
    args.num_radial = 6
    args.envelope_exponent = 5
    args.num_before_skip = 1
    args.num_after_skip = 2
    args.num_output_layers = 3
    args.use_node_features = True

    # Learning
    args.p_epoch = 100
    args.p_lr = 1e-3
    args.aug_1, args.aug_2 = 'MMFFrandom', 'MMFFrandom'
    args.aug_ratio = 0.25
    args.tau = 0.2
    args.proj = 'spherenet'

    # Regularization
    args.dropout_rate = 0.0

    args.p_optim = 'ExponentialLR' #['StepLR', ExponentialLR, 'Cosine']
    
    #'StepLR'
    args.p_weight_decay = 0
    args.p_lr_decay_step_size = 15  # 15 epoch 마다 lr * p_lr_decay_factor
    args.p_lr_decay_factor = 0.5

    # ExponentialLR
    args.expo_gamma = 0.95
    
    # Cosine
    args.T_0 = 20        # 최초 주기값
    args.T_mult = 2      # 최초 주기값에 비해 얼만큼 주기를 늘려갈 것인지
    args.eta_max = 0.05  # lr 최대값
    args.T_up = 10      # Warm up 시 필요한 epoch 수(일반적으로 짧은 수)
    args.gamma = 0.5     # 주기가 반복될수록 곱해지는 scale 값

    args.pc = False

    encoder = Encoder(args)
    graphcl = GraphCL(args)
    evaluator = Pretrain(args)
    encoder = evaluator.evaluate(learning_model=graphcl, encoder=encoder)


main()

args.device: xpu:0


Pretraining: epoch 1:   0%|          | 0/100 [00:00<?, ?it/s]

./models/encoder-schnet_pretrain-esol_batch-400_proj-spherenet_cutoff-5.0_layers-2_filter-128_gau-50_z_dim-32_lr-0.001_aug_1-MMFFrandom_aug_2-MMFFrandom_aug_ratio-0.25_tau-0.2_optim-ExponentialLR_weight_decay-0_expo_gamma-0.95_dropout-0.0


c:\DGCL\3DGCL\examples\sslgraph\../..\dig\sslgraph\method\contrastive\model\contrastive.py:153: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  t.set_postfix(loss='{:.4f}'.format(float(epoch_loss)))
Pretraining: epoch 29:  28%|██▊       | 28/100 [01:21<03:33,  2.96s/it, loss=9.4066] 

In [5]:
# pretrain에서 생성된 최신 체크포인트 경로를 CSV로 저장/확인
from pathlib import Path
import pandas as pd

models_root = Path("./models")
latest = sorted(
    models_root.rglob("enc_best_epoch-*_loss-*.pkl"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
if not latest:
    print("No checkpoint found under ./models")
else:
    latest_ckpt = latest[0].resolve()
    row = {
        "checkpoint_path": str(latest_ckpt),
        "checkpoint_name": latest_ckpt.name,
        "modified_time": latest_ckpt.stat().st_mtime,
    }
    out_csv = models_root / "latest_checkpoint.csv"
    pd.DataFrame([row]).to_csv(out_csv, index=False)
    print(f"latest checkpoint: {latest_ckpt}")
    print(f"saved csv: {out_csv.resolve()}")

latest checkpoint: C:\DGCL\3DGCL\examples\sslgraph\models\encoder-schnet_pretrain-esol_batch-400_proj-spherenet_cutoff-5.0_layers-2_filter-128_gau-50_z_dim-32_lr-0.001_aug_1-MMFFrandom_aug_2-MMFFrandom_aug_ratio-0.25_tau-0.2_optim-ExponentialLR_weight_decay-0_expo_gamma-0.95_dropout-0.0\enc_best_epoch-82_loss-9.112.pkl
saved csv: C:\DGCL\3DGCL\examples\sslgraph\models\latest_checkpoint.csv
